In [1]:
import pandas as pd

train = pd.read_csv('household_train.csv')
valid = pd.read_csv('household_validation.csv')
test = pd.read_csv('household_test.csv')

In [ ]:
input_width_30 = 30
label_width_30 = 30

input_width_24 = 48
label_width_24 = 24

feature_columns = [
    "Global_active_power", "Global_intensity",
    "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]

In [ ]:
import numpy as np
import tensorflow as tf

def create_sequences(data, input_width, label_width, feature_columns, target_column="Global_active_power"):
    X, y = [], []
    values_X = data[feature_columns].values
    values_y = data[target_column].values
    for i in range(len(data) - input_width - label_width):
        X.append(values_X[i:i+input_width])
        y.append(values_y[i+input_width:i+input_width+label_width])
    return np.array(X)[..., np.newaxis], np.array(y)

X_train, y_train = create_sequences(train, input_width_30, label_width_30, feature_columns)
X_valid, y_valid = create_sequences(valid, input_width_30, label_width_30, feature_columns)
X_test,  y_test  = create_sequences(test,  input_width_30, label_width_30, feature_columns)


2025-09-07 11:46:00.847353: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Tuning

In [4]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
import random

def build_model(input_width, n_features, label_width,
                filters, kernel_size, dense_units, dropout_rate, learning_rate,
                use_pooling=True):
    model = Sequential()
    model.add(Conv1D(filters=filters, kernel_size=kernel_size,
                     activation='relu',
                     input_shape=(input_width, n_features)))
    
    if use_pooling:
        model.add(MaxPooling1D(pool_size=2))
    
    model.add(Flatten())
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(label_width))
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss="mse", metrics=["mae"])
    return model


In [ ]:
param_grid = {
    "filters": [16, 32, 64],
    "kernel_size": [2, 3, 5],
    "dense_units": [32, 64, 128],
    "dropout_rate": [0.0, 0.2, 0.5],
    "learning_rate": [0.002, 0.001, 0.0005],
    "batch_size": [32, 64],
    "use_pooling": [True, False]
}

n_trials = 50

results_30 = []

for i in range(n_trials):
    params = {k: random.choice(v) for k, v in param_grid.items()}
    print(f"Trial {i+1}: {params}")

    batch_size = params.pop("batch_size")

    model = build_model(input_width_30, len(feature_columns), label_width_30, **params)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_valid, y_valid),
        epochs=4,
        batch_size=batch_size,
        verbose=1
    )
    val_mae = min(history.history["val_mae"])
    results_30.append((params | {"batch_size": batch_size}, val_mae))

best_params, best_score = sorted(results_30, key=lambda x: x[1])[0]
print("Beste Kombination:", best_params, "mit val_mae:", best_score)


## 24 Stunden

In [ ]:
train_hourly = train[feature_columns].resample('H').mean()
valid_hourly = valid[feature_columns].resample('H').mean()
test_hourly  = test[feature_columns].resample('H').mean()

X_train_24, y_train_24 = create_sequences(train_hourly, input_width_24, label_width_24, feature_columns)
X_valid_24, y_valid_24 = create_sequences(valid_hourly, input_width_24, label_width_24, feature_columns)
X_test_24,  y_test_24  = create_sequences(test_hourly,  input_width_24, label_width_24, feature_columns)


In [ ]:
param_grid = {
    "filters": [16, 32, 64],
    "kernel_size": [2, 3, 5],
    "dense_units": [32, 64, 128],
    "dropout_rate": [0.0, 0.2, 0.5],
    "learning_rate": [0.002, 0.001, 0.0005],
    "batch_size": [32, 64],
    "use_pooling": [True, False]
}

n_trials = 50

results_24 = []

for i in range(n_trials):
    params = {k: random.choice(v) for k, v in param_grid.items()}
    print(f"Trial {i+1}: {params}")

    batch_size = params.pop("batch_size")

    model = build_model(input_width_24, len(feature_columns), label_width_24, **params)

    history = model.fit(
        X_train_24, y_train_24,
        validation_data=(X_valid_24, y_valid_24),
        epochs=50,
        batch_size=batch_size,
        verbose=1
    )
    val_mae = min(history.history["val_mae"])
    results_24.append((params | {"batch_size": batch_size}, val_mae))

best_params, best_score = sorted(results_24, key=lambda x: x[1])[0]
print("Beste Kombination:", best_params, "mit val_mae:", best_score)
